# Advanced Tutorial Problems with Solutions
## Updating, Merging, Unpacking, and Copying Python Dictionaries

This notebook is a **second, independent problem set** on the same topic.

The style is intentionally tutorial-oriented:

- each problem starts with a practical scenario;
- the reasoning is divided into small logical steps;
- important predictions are discussed before code is run;
- solutions are built incrementally rather than presented all at once;
- assertions and edge cases are included to verify the result.

The notebook assumes familiarity with basic dictionary syntax, but it explains the advanced behavior carefully.

## Learning goals

By the end of the notebook, you should be able to:

1. reason precisely about overwrite precedence and insertion order;
2. update dictionaries from mappings, pair iterables, generators, and keyword arguments;
3. merge partial records without mutating source data;
4. choose different merge policies for different fields;
5. detect shared mutable objects created by shallow copying;
6. create deliberately isolated snapshots;
7. perform a three-way merge and report conflicts;
8. generate reversible dictionary patches;
9. compute and replay dictionary differences;
10. update nested dictionaries from dotted paths;
11. combine keyword-argument providers safely;
12. build a small versioned settings store with rollback support.

## A compact refresher

The core operations in this notebook are:

- `d[key] = value` for one assignment;
- `d.update(other)` for an in-place update;
- `{**left, **right}` for creating a new merged dictionary;
- `left | right` for creating a new merged dictionary in modern Python;
- `d.copy()` for a shallow copy;
- `copy.deepcopy(d)` for a recursive copy.

A crucial rule is that when the same key appears more than once, the **last value wins**.

However, overwriting an existing key does **not** move that key to the end of the dictionary.

In [1]:
from copy import copy, deepcopy
from collections.abc import Mapping, Iterable
from dataclasses import dataclass
from typing import Any

print("Setup complete.")

Setup complete.


# Problem 1 — Ordered overwrites in layered settings

A command-line application receives settings from three places:

1. built-in defaults;
2. a project configuration file;
3. command-line overrides.

The final values should come from the last layer that mentions a key. At the same time, the display order should remain predictable.

## Starting data

Before writing any helper function, inspect the three layers below.

In [2]:
defaults = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "workers": 1,
}

project = {
    "port": 8080,
    "workers": 4,
    "log_level": "INFO",
}

cli = {
    "debug": True,
    "port": 9000,
}

## Step 1 — Predict the values

The precedence is:

```text
defaults < project < cli
```

Therefore:

- `port` should end as `9000`;
- `workers` should end as `4`;
- `debug` should end as `True`;
- `log_level` should be added by the project layer.

Now consider the order. The original default keys keep their original positions. The new key `log_level` is appended when it first appears.

## Step 2 — Build the result without mutating the inputs

A clean approach is to make a shallow copy of the first layer and update that copy.

In [3]:
def merge_layers(*layers):
    """Return a new dictionary using left-to-right overwrite precedence."""
    result = {}
    for layer in layers:
        result.update(layer)
    return result

settings = merge_layers(defaults, project, cli)
settings

{'host': 'localhost',
 'port': 9000,
 'debug': True,
 'workers': 4,
 'log_level': 'INFO'}

## Step 3 — Verify both values and order

Checking only equality is not enough when order is part of the requirement. We should also inspect the key sequence explicitly.

In [4]:
assert settings == {
    "host": "localhost",
    "port": 9000,
    "debug": True,
    "workers": 4,
    "log_level": "INFO",
}

assert list(settings) == ["host", "port", "debug", "workers", "log_level"]
assert defaults["port"] == 8000
assert project["port"] == 8080

print("Final settings:", settings)
print("Key order:", list(settings))

Final settings: {'host': 'localhost', 'port': 9000, 'debug': True, 'workers': 4, 'log_level': 'INFO'}
Key order: ['host', 'port', 'debug', 'workers', 'log_level']


## Step 4 — Track which keys were overwritten

In real systems, it is often useful to explain why a final value was selected. We can record an overwrite event each time a later layer replaces a key.

In [5]:
def merge_layers_with_history(*named_layers):
    """Merge `(name, mapping)` layers and record every overwrite."""
    result = {}
    owner = {}
    history = []

    for layer_name, layer in named_layers:
        for key, value in layer.items():
            if key in result:
                history.append({
                    "key": key,
                    "old_value": result[key],
                    "new_value": value,
                    "old_layer": owner[key],
                    "new_layer": layer_name,
                })
            result[key] = value
            owner[key] = layer_name

    return result, history

settings2, history = merge_layers_with_history(
    ("defaults", defaults),
    ("project", project),
    ("cli", cli),
)

settings2, history

({'host': 'localhost',
  'port': 9000,
  'debug': True,
  'workers': 4,
  'log_level': 'INFO'},
 [{'key': 'port',
   'old_value': 8000,
   'new_value': 8080,
   'old_layer': 'defaults',
   'new_layer': 'project'},
  {'key': 'workers',
   'old_value': 1,
   'new_value': 4,
   'old_layer': 'defaults',
   'new_layer': 'project'},
  {'key': 'debug',
   'old_value': False,
   'new_value': True,
   'old_layer': 'defaults',
   'new_layer': 'cli'},
  {'key': 'port',
   'old_value': 8080,
   'new_value': 9000,
   'old_layer': 'project',
   'new_layer': 'cli'}])

## Solution discussion

The function uses ordinary assignment rather than any special merge structure. This is useful because it makes the overwrite rule explicit.

Notice that assigning a new value to an existing key changes the value but keeps the key's position. A key is appended only when it is seen for the first time.

# Problem 2 — Updating from heterogeneous sources

The `update` method can consume several source forms. Suppose a program receives configuration fragments as:

- a normal dictionary;
- a list of `(key, value)` pairs;
- a generator of pairs;
- final keyword arguments.

Create one helper that accepts all of these forms and returns a new dictionary.

## Step 1 — Prepare different source types

All sources below ultimately describe key-value pairs, but they are represented differently.

In [6]:
base = {"alpha": 1, "beta": 2}
source_mapping = {"beta": 20, "gamma": 30}
source_pairs = [("delta", 40), ("epsilon", 50)]
source_generator = ((letter, ord(letter)) for letter in "xy")

## Step 2 — Normalize each source with `dict(...)`

The constructor `dict(source)` is a convenient validation and normalization step:

- mappings become ordinary dictionaries;
- pair iterables become dictionaries;
- generators are consumed once;
- malformed elements raise a useful exception.

Because the generator is one-shot, it should not be reused after normalization.

In [7]:
def updated_copy(base, *sources, **keyword_updates):
    """Return a shallowly copied and updated dictionary."""
    result = base.copy()

    for source in sources:
        normalized = dict(source)
        result.update(normalized)

    result.update(keyword_updates)
    return result

combined = updated_copy(
    base,
    source_mapping,
    source_pairs,
    source_generator,
    beta=200,
    zeta=60,
)

combined

{'alpha': 1,
 'beta': 200,
 'gamma': 30,
 'delta': 40,
 'epsilon': 50,
 'x': 120,
 'y': 121,
 'zeta': 60}

## Step 3 — Explain the final precedence

The order of application is:

1. `base` is copied;
2. `source_mapping` changes `beta` to `20`;
3. the pair list adds `delta` and `epsilon`;
4. the generator adds `x` and `y`;
5. keyword arguments change `beta` again to `200` and add `zeta`.

Therefore the keyword argument wins for `beta`.

In [8]:
assert combined["beta"] == 200
assert combined["gamma"] == 30
assert combined["delta"] == 40
assert combined["x"] == ord("x")
assert combined["zeta"] == 60
assert base == {"alpha": 1, "beta": 2}

print(combined)

{'alpha': 1, 'beta': 200, 'gamma': 30, 'delta': 40, 'epsilon': 50, 'x': 120, 'y': 121, 'zeta': 60}


## Step 4 — Demonstrate malformed input safely

An element with three values cannot represent a key-value pair. We catch the exception so the notebook continues running.

In [9]:
bad_source = [("a", 1, "extra")]

try:
    dict(bad_source)
except (TypeError, ValueError) as exc:
    print(type(exc).__name__ + ":", exc)

ValueError: dictionary update sequence element #0 has length 3; 2 is required


## Solution discussion

This helper intentionally returns a new dictionary rather than mutating `base`. The copy is shallow, so nested mutable values would still be shared. Later problems examine that issue directly.

# Problem 3 — Reconcile partial product records

An inventory system receives a list of complete product records and then a stream of partial updates. Each update contains a `sku` and one or more fields to replace.

Requirements:

- keep the original product order;
- reject updates for unknown products;
- reject unknown fields;
- do not mutate the original records;
- allow more than one update for the same product.

## Step 1 — Starting records and updates

In [10]:
products = [
    {"sku": "A100", "name": "Keyboard", "price": 50.0, "stock": 10},
    {"sku": "B200", "name": "Mouse", "price": 25.0, "stock": 30},
    {"sku": "C300", "name": "Monitor", "price": 220.0, "stock": 8},
]

updates = [
    {"sku": "B200", "stock": 28},
    {"sku": "A100", "price": 47.5},
    {"sku": "B200", "price": 23.0, "stock": 35},
]

## Step 2 — Create a copied index

A dictionary indexed by SKU gives constant-time lookup. We copy every product dictionary so updates do not affect the originals.

In [11]:
def build_product_index(records):
    index = {}
    order = []

    for record in records:
        sku = record["sku"]
        if sku in index:
            raise ValueError(f"duplicate SKU: {sku}")
        index[sku] = record.copy()
        order.append(sku)

    return index, order

product_index, product_order = build_product_index(products)
product_index

{'A100': {'sku': 'A100', 'name': 'Keyboard', 'price': 50.0, 'stock': 10},
 'B200': {'sku': 'B200', 'name': 'Mouse', 'price': 25.0, 'stock': 30},
 'C300': {'sku': 'C300', 'name': 'Monitor', 'price': 220.0, 'stock': 8}}

## Step 3 — Validate update fields before applying them

The allowed fields come from the existing record. The `sku` field identifies the record and should not be replaced.

In [12]:
def reconcile_products(records, updates):
    index, order = build_product_index(records)

    for patch in updates:
        if "sku" not in patch:
            raise ValueError("every update must include 'sku'")

        sku = patch["sku"]
        if sku not in index:
            raise KeyError(f"unknown SKU: {sku}")

        allowed = set(index[sku]) - {"sku"}
        supplied = set(patch) - {"sku"}
        unknown = supplied - allowed

        if unknown:
            raise ValueError(f"unknown fields for {sku}: {sorted(unknown)}")

        changes = {key: patch[key] for key in supplied}
        index[sku].update(changes)

    return [index[sku] for sku in order]

reconciled = reconcile_products(products, updates)
reconciled

[{'sku': 'A100', 'name': 'Keyboard', 'price': 47.5, 'stock': 10},
 {'sku': 'B200', 'name': 'Mouse', 'price': 23.0, 'stock': 35},
 {'sku': 'C300', 'name': 'Monitor', 'price': 220.0, 'stock': 8}]

## Step 4 — Verify order, repeated updates, and non-mutation

In [13]:
assert [item["sku"] for item in reconciled] == ["A100", "B200", "C300"]
assert reconciled[0]["price"] == 47.5
assert reconciled[1]["price"] == 23.0
assert reconciled[1]["stock"] == 35
assert products[0]["price"] == 50.0
assert products[1]["stock"] == 30

print("Original:", products)
print("Reconciled:", reconciled)

Original: [{'sku': 'A100', 'name': 'Keyboard', 'price': 50.0, 'stock': 10}, {'sku': 'B200', 'name': 'Mouse', 'price': 25.0, 'stock': 30}, {'sku': 'C300', 'name': 'Monitor', 'price': 220.0, 'stock': 8}]
Reconciled: [{'sku': 'A100', 'name': 'Keyboard', 'price': 47.5, 'stock': 10}, {'sku': 'B200', 'name': 'Mouse', 'price': 23.0, 'stock': 35}, {'sku': 'C300', 'name': 'Monitor', 'price': 220.0, 'stock': 8}]


## Step 5 — Test an invalid field

In [14]:
try:
    reconcile_products(products, [{"sku": "A100", "color": "black"}])
except ValueError as exc:
    print(exc)

unknown fields for A100: ['color']


## Solution discussion

The important design choice is to separate **lookup order** from **output order**:

- the dictionary index makes updates efficient;
- the separate `order` list preserves the first-seen record order.

The function performs shallow copies because every field in this example is immutable. If records contained nested lists or dictionaries, a deeper copying strategy might be needed.

# Problem 4 — Merge fields using different policies

Plain `dict.update` always replaces an old value with a new value. That is not always the desired meaning.

Suppose two analytics dictionaries contain:

- numeric counters that should be added;
- tag sets that should be unioned;
- timestamps that should keep the maximum value;
- descriptive text that should use normal last-value-wins behavior.

## Step 1 — Observe why plain `update` is insufficient

In [15]:
left_metrics = {
    "views": 120,
    "clicks": 12,
    "tags": {"python", "dict"},
    "last_seen": 1000,
    "note": "morning batch",
}

right_metrics = {
    "views": 80,
    "clicks": 9,
    "tags": {"copying", "dict"},
    "last_seen": 1200,
    "note": "afternoon batch",
}

plain = left_metrics.copy()
plain.update(right_metrics)
plain

{'views': 80,
 'clicks': 9,
 'tags': {'copying', 'dict'},
 'last_seen': 1200,
 'note': 'afternoon batch'}

The plain update loses the first batch's counters and tags. We need a policy table.

## Step 2 — Define policy functions

In [16]:
def add_values(old, new):
    return old + new

def union_values(old, new):
    return old | new

def max_value(old, new):
    return max(old, new)

def replace_value(old, new):
    return new

POLICIES = {
    "views": add_values,
    "clicks": add_values,
    "tags": union_values,
    "last_seen": max_value,
    "note": replace_value,
}

## Step 3 — Apply the appropriate policy per key

For keys that appear only on one side, simply copy the available value. For keys present on both sides, call the selected policy.

In [17]:
def merge_with_policies(left, right, policies):
    result = left.copy()

    for key, new_value in right.items():
        if key not in result:
            result[key] = deepcopy(new_value)
            continue

        policy = policies.get(key, replace_value)
        result[key] = policy(result[key], new_value)

    return result

merged_metrics = merge_with_policies(left_metrics, right_metrics, POLICIES)
merged_metrics

{'views': 200,
 'clicks': 21,
 'tags': {'copying', 'dict', 'python'},
 'last_seen': 1200,
 'note': 'afternoon batch'}

## Step 4 — Verify the semantics and copying behavior

In [18]:
assert merged_metrics["views"] == 200
assert merged_metrics["clicks"] == 21
assert merged_metrics["tags"] == {"python", "dict", "copying"}
assert merged_metrics["last_seen"] == 1200
assert merged_metrics["note"] == "afternoon batch"

merged_metrics["tags"].add("advanced")
assert "advanced" not in left_metrics["tags"]
assert "advanced" not in right_metrics["tags"]

print(merged_metrics)

{'views': 200, 'clicks': 21, 'tags': {'dict', 'advanced', 'copying', 'python'}, 'last_seen': 1200, 'note': 'afternoon batch'}


## Solution discussion

A merge operation has a meaning, not just a syntax. `update` is correct when replacement is the intended meaning. When different fields represent different concepts, a policy-driven merge is clearer and safer.

# Problem 5 — Detect shared mutable values after a shallow copy

A shallow copy creates a new outer dictionary, but nested objects remain shared.

Instead of merely demonstrating this fact, build a diagnostic tool that finds shared mutable objects between two dictionary graphs.

## Step 1 — Create a shallow copy and mutate a nested object

In [19]:
original = {
    "profile": {"name": "Ada", "skills": ["math", "code"]},
    "flags": {"active", "verified"},
    "score": 100,
}

shallow = original.copy()
shallow["profile"]["skills"].append("logic")

print("original:", original)
print("shallow:", shallow)

original: {'profile': {'name': 'Ada', 'skills': ['math', 'code', 'logic']}, 'flags': {'active', 'verified'}, 'score': 100}
shallow: {'profile': {'name': 'Ada', 'skills': ['math', 'code', 'logic']}, 'flags': {'active', 'verified'}, 'score': 100}


The mutation appears in both dictionaries because the nested list is the same object.

We now want a function that reports shared mutable objects and the paths where they occur.

## Step 2 — Traverse nested containers

We will treat dictionaries, lists, sets, and byte arrays as mutable containers. Tuples are traversed because they may contain mutable objects, even though the tuple itself is immutable.

In [20]:
MUTABLE_TYPES = (dict, list, set, bytearray)
TRAVERSABLE_TYPES = (dict, list, tuple, set)

def walk_objects(value, path=()):
    """Yield `(path, object)` pairs while avoiding infinite recursion."""
    seen = set()

    def walk(current, current_path):
        object_id = id(current)
        if object_id in seen:
            return
        seen.add(object_id)

        yield current_path, current

        if isinstance(current, dict):
            for key, child in current.items():
                yield from walk(child, current_path + (key,))
        elif isinstance(current, (list, tuple)):
            for index, child in enumerate(current):
                yield from walk(child, current_path + (index,))
        elif isinstance(current, set):
            for index, child in enumerate(sorted(current, key=repr)):
                yield from walk(child, current_path + (f"set[{index}]",))

    yield from walk(value, path)

## Step 3 — Compare object identities

Two paths are shared when they point to the same mutable object, which we test with object identity rather than equality.

In [21]:
def find_shared_mutables(left, right):
    left_objects = list(walk_objects(left))
    right_by_id = {}

    for path, obj in walk_objects(right):
        right_by_id.setdefault(id(obj), []).append(path)

    shared = []
    for left_path, obj in left_objects:
        if isinstance(obj, MUTABLE_TYPES) and id(obj) in right_by_id:
            for right_path in right_by_id[id(obj)]:
                shared.append({
                    "left_path": left_path,
                    "right_path": right_path,
                    "type": type(obj).__name__,
                })

    return shared

shared_report = find_shared_mutables(original, shallow)
shared_report

[{'left_path': ('profile',), 'right_path': ('profile',), 'type': 'dict'},
 {'left_path': ('profile', 'skills'),
  'right_path': ('profile', 'skills'),
  'type': 'list'},
 {'left_path': ('flags',), 'right_path': ('flags',), 'type': 'set'}]

## Step 4 — Compare with a deep copy

In [22]:
isolated = deepcopy(original)
shared_with_deepcopy = find_shared_mutables(original, isolated)

assert shared_report
assert shared_with_deepcopy == []

print("Shared after shallow copy:")
for item in shared_report:
    print(item)

print("\nShared after deep copy:", shared_with_deepcopy)

Shared after shallow copy:
{'left_path': ('profile',), 'right_path': ('profile',), 'type': 'dict'}
{'left_path': ('profile', 'skills'), 'right_path': ('profile', 'skills'), 'type': 'list'}
{'left_path': ('flags',), 'right_path': ('flags',), 'type': 'set'}

Shared after deep copy: []


## Solution discussion

The diagnostic is based on `id(...)`, not on `==`. Two equal lists may be independent objects, while one shared list has exactly one identity.

The traversal also keeps a `seen` set so cyclic structures cannot cause infinite recursion.

# Problem 6 — Build an isolated public snapshot

An application stores internal state containing both public data and sensitive data. A function should return a public snapshot that callers may freely mutate without affecting internal state.

Requirements:

- omit secret fields;
- preserve only selected public fields;
- deeply isolate the returned data;
- leave the internal state untouched.

## Step 1 — Internal state

In [23]:
internal_state = {
    "user": {
        "id": 42,
        "name": "Grace",
        "preferences": {
            "theme": "dark",
            "languages": ["Python", "C"],
        },
    },
    "session": {
        "token": "SECRET-TOKEN",
        "scopes": ["read", "write"],
    },
    "statistics": {
        "logins": 17,
        "recent_ips": ["10.0.0.1", "10.0.0.2"],
    },
}

## Step 2 — Select before copying

A useful best practice is to construct the smallest permitted structure first, then deep-copy it. This avoids accidentally exposing a field merely because it existed in the original dictionary.

In [24]:
def public_snapshot(state):
    selected = {
        "user": state["user"],
        "statistics": state["statistics"],
    }
    return deepcopy(selected)

snapshot = public_snapshot(internal_state)
snapshot

{'user': {'id': 42,
  'name': 'Grace',
  'preferences': {'theme': 'dark', 'languages': ['Python', 'C']}},
 'statistics': {'logins': 17, 'recent_ips': ['10.0.0.1', '10.0.0.2']}}

## Step 3 — Mutate the snapshot aggressively

These mutations should not leak back into `internal_state`.

In [25]:
snapshot["user"]["name"] = "Changed Name"
snapshot["user"]["preferences"]["languages"].append("Rust")
snapshot["statistics"]["recent_ips"].clear()
snapshot["extra"] = True

assert internal_state["user"]["name"] == "Grace"
assert internal_state["user"]["preferences"]["languages"] == ["Python", "C"]
assert internal_state["statistics"]["recent_ips"] == ["10.0.0.1", "10.0.0.2"]
assert "session" not in snapshot

print("snapshot:", snapshot)
print("internal:", internal_state)

snapshot: {'user': {'id': 42, 'name': 'Changed Name', 'preferences': {'theme': 'dark', 'languages': ['Python', 'C', 'Rust']}}, 'statistics': {'logins': 17, 'recent_ips': []}, 'extra': True}
internal: {'user': {'id': 42, 'name': 'Grace', 'preferences': {'theme': 'dark', 'languages': ['Python', 'C']}}, 'session': {'token': 'SECRET-TOKEN', 'scopes': ['read', 'write']}, 'statistics': {'logins': 17, 'recent_ips': ['10.0.0.1', '10.0.0.2']}}


## Step 4 — Why a shallow copy would fail

A shallow copy of the selected outer dictionary would still share the nested `user` and `statistics` dictionaries. Deep copying is appropriate because the snapshot is an isolation boundary.

# Problem 7 — Three-way merge with conflict detection

Three-way merging is used when two editors begin from the same base dictionary.

Given:

- `base`: the common starting point;
- `ours`: our edited version;
- `theirs`: another edited version;

merge non-conflicting changes automatically and report conflicting changes.

## Step 1 — Understand the cases for one key

For any key:

1. neither side changed it → keep the base value;
2. only ours changed it → use ours;
3. only theirs changed it → use theirs;
4. both changed it to the same value → use that value;
5. both changed it differently → report a conflict.

Additions and deletions must also be represented, so we need a private sentinel for a missing key.

In [26]:
MISSING = object()

base_doc = {
    "title": "Dictionary Notes",
    "status": "draft",
    "reviewer": "Mina",
    "pages": 5,
}

ours_doc = {
    "title": "Advanced Dictionary Notes",
    "status": "draft",
    "pages": 6,
    "author_note": "Expanded examples",
}

theirs_doc = {
    "title": "Dictionary Notes",
    "status": "review",
    "reviewer": "Mina",
    "pages": 7,
}

## Step 2 — Write a helper for one key

In [27]:
def merge_one_value(base_value, our_value, their_value):
    our_changed = our_value != base_value
    their_changed = their_value != base_value

    if not our_changed and not their_changed:
        return "merged", base_value
    if our_changed and not their_changed:
        return "merged", our_value
    if not our_changed and their_changed:
        return "merged", their_value
    if our_value == their_value:
        return "merged", our_value
    return "conflict", {
        "base": base_value,
        "ours": our_value,
        "theirs": their_value,
    }

## Step 3 — Extend the logic to all keys

The union of key sets ensures that additions and deletions are considered.

In [28]:
def three_way_merge(base, ours, theirs):
    merged = {}
    conflicts = {}

    all_keys = set(base) | set(ours) | set(theirs)

    for key in all_keys:
        status, result = merge_one_value(
            base.get(key, MISSING),
            ours.get(key, MISSING),
            theirs.get(key, MISSING),
        )

        if status == "conflict":
            conflicts[key] = result
        elif result is not MISSING:
            merged[key] = deepcopy(result)

    return merged, conflicts

merged_doc, conflicts = three_way_merge(base_doc, ours_doc, theirs_doc)
merged_doc, conflicts

({'status': 'review',
  'title': 'Advanced Dictionary Notes',
  'author_note': 'Expanded examples'},
 {'pages': {'base': 5, 'ours': 6, 'theirs': 7}})

## Step 4 — Interpret the result

- `title` changed only in ours, so it is merged.
- `status` changed only in theirs, so it is merged.
- `reviewer` was deleted only in ours, so the deletion is merged.
- `author_note` was added only in ours, so it is merged.
- `pages` changed differently on both sides, so it is a conflict.

In [29]:
assert merged_doc["title"] == "Advanced Dictionary Notes"
assert merged_doc["status"] == "review"
assert merged_doc["author_note"] == "Expanded examples"
assert "reviewer" not in merged_doc
assert set(conflicts) == {"pages"}
assert conflicts["pages"] == {"base": 5, "ours": 6, "theirs": 7}

print("Merged portion:", merged_doc)
print("Conflicts:", conflicts)

Merged portion: {'status': 'review', 'title': 'Advanced Dictionary Notes', 'author_note': 'Expanded examples'}
Conflicts: {'pages': {'base': 5, 'ours': 6, 'theirs': 7}}


## Step 5 — Resolve a conflict explicitly

A conflict should not be silently guessed. Here we choose the larger page count as a domain-specific resolution.

In [30]:
resolved_doc = merged_doc.copy()
resolved_doc["pages"] = max(
    conflicts["pages"]["ours"],
    conflicts["pages"]["theirs"],
)

resolved_doc

{'status': 'review',
 'title': 'Advanced Dictionary Notes',
 'author_note': 'Expanded examples',
 'pages': 7}

## Solution discussion

Three-way merging is fundamentally different from simply merging `ours` and `theirs`. The base version tells us whether a side actually changed a value.

# Problem 8 — Apply a patch and generate its inverse

A patch changes selected keys. To support undo, the operation should also return an inverse patch.

We will support two operations:

- `("set", key, value)`
- `("delete", key)`

The function should return a new dictionary and a reverse patch that restores the original state.

## Step 1 — Define the patch

In [31]:
state = {
    "theme": "light",
    "font_size": 14,
    "sidebar": True,
}

patch = [
    ("set", "theme", "dark"),
    ("set", "language", "en"),
    ("delete", "sidebar"),
]

## Step 2 — Record inverse operations before applying changes

The inverse of setting an existing key is another set operation restoring the old value.

The inverse of setting a new key is a delete operation.

The inverse of deleting an existing key is a set operation restoring the deleted value.

In [32]:
def apply_patch_with_inverse(mapping, operations):
    result = deepcopy(mapping)
    inverse = []

    for operation in operations:
        kind = operation[0]

        if kind == "set":
            _, key, new_value = operation
            if key in result:
                inverse.append(("set", key, deepcopy(result[key])))
            else:
                inverse.append(("delete", key))
            result[key] = deepcopy(new_value)

        elif kind == "delete":
            _, key = operation
            if key not in result:
                raise KeyError(f"cannot delete missing key: {key}")
            inverse.append(("set", key, deepcopy(result[key])))
            del result[key]

        else:
            raise ValueError(f"unknown operation: {kind}")

    inverse.reverse()
    return result, inverse

patched, inverse_patch = apply_patch_with_inverse(state, patch)
patched, inverse_patch

({'theme': 'dark', 'font_size': 14, 'language': 'en'},
 [('set', 'sidebar', True), ('delete', 'language'), ('set', 'theme', 'light')])

## Step 3 — Apply the inverse patch

The inverse operations are reversed because undo must happen in the opposite order.

In [33]:
restored, redo_patch = apply_patch_with_inverse(patched, inverse_patch)

assert restored == state
assert state == {
    "theme": "light",
    "font_size": 14,
    "sidebar": True,
}

print("Patched:", patched)
print("Inverse:", inverse_patch)
print("Restored:", restored)

Patched: {'theme': 'dark', 'font_size': 14, 'language': 'en'}
Inverse: [('set', 'sidebar', True), ('delete', 'language'), ('set', 'theme', 'light')]
Restored: {'theme': 'light', 'font_size': 14, 'sidebar': True}


## Step 4 — Why values are deep-copied

If a patch value were a list or dictionary, storing the same object inside both the result and the inverse patch could allow later mutations to corrupt the undo history. Deep copies make the history stable.

# Problem 9 — Compute and replay a dictionary difference

A difference describes how to transform one dictionary into another.

Produce four categories:

- `added` keys;
- `removed` keys;
- `changed` keys with old and new values;
- `unchanged` keys.

Then write a function that applies the difference to reconstruct the target dictionary.

## Step 1 — Example versions

In [34]:
version_a = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "workers": 2,
}

version_b = {
    "host": "localhost",
    "port": 9000,
    "debug": True,
    "timeout": 30,
}

## Step 2 — Classify keys

In [35]:
def dict_diff(old, new):
    old_keys = set(old)
    new_keys = set(new)

    added = {key: deepcopy(new[key]) for key in new_keys - old_keys}
    removed = {key: deepcopy(old[key]) for key in old_keys - new_keys}

    shared = old_keys & new_keys
    changed = {
        key: {"old": deepcopy(old[key]), "new": deepcopy(new[key])}
        for key in shared
        if old[key] != new[key]
    }
    unchanged = {
        key: deepcopy(old[key])
        for key in shared
        if old[key] == new[key]
    }

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
        "unchanged": unchanged,
    }

difference = dict_diff(version_a, version_b)
difference

{'added': {'timeout': 30},
 'removed': {'workers': 2},
 'changed': {'port': {'old': 8000, 'new': 9000},
  'debug': {'old': False, 'new': True}},
 'unchanged': {'host': 'localhost'}}

## Step 3 — Replay the difference

To transform the old version:

1. remove the removed keys;
2. set the new values of changed keys;
3. add the added keys.

In [36]:
def apply_diff(old, difference):
    result = deepcopy(old)

    for key in difference["removed"]:
        result.pop(key, None)

    for key, values in difference["changed"].items():
        result[key] = deepcopy(values["new"])

    for key, value in difference["added"].items():
        result[key] = deepcopy(value)

    return result

replayed = apply_diff(version_a, difference)
replayed

{'host': 'localhost', 'port': 9000, 'debug': True, 'timeout': 30}

## Step 4 — Verify and inspect

In [37]:
assert replayed == version_b
assert difference["added"] == {"timeout": 30}
assert difference["removed"] == {"workers": 2}
assert difference["changed"]["port"] == {"old": 8000, "new": 9000}
assert difference["unchanged"] == {"host": "localhost"}

for category, content in difference.items():
    print(f"{category}: {content}")

added: {'timeout': 30}
removed: {'workers': 2}
changed: {'port': {'old': 8000, 'new': 9000}, 'debug': {'old': False, 'new': True}}
unchanged: {'host': 'localhost'}


## Solution discussion

A difference is more informative than a merged dictionary because it explains what changed. It can be logged, reviewed, tested, or converted into a patch.

# Problem 10 — Update nested dictionaries from dotted paths

Configuration tools often express nested keys as strings such as:

```text
"database.host"
"database.credentials.user"
```

Implement an updater that accepts dotted paths and returns a new deeply isolated dictionary.

## Step 1 — Starting configuration and dotted updates

In [38]:
configuration = {
    "database": {
        "host": "localhost",
        "port": 5432,
        "credentials": {
            "user": "app",
            "password": "dev-password",
        },
    },
    "features": {
        "search": False,
    },
}

dotted_updates = {
    "database.host": "db.internal",
    "database.credentials.user": "service",
    "features.search": True,
    "features.analytics": True,
}

## Step 2 — Update one path

The path is split into components. All intermediate components must refer to dictionaries. The final component is assigned the new value.

In [39]:
def set_dotted_path(mapping, dotted_path, value):
    parts = dotted_path.split(".")
    if not all(parts):
        raise ValueError(f"invalid dotted path: {dotted_path!r}")

    current = mapping
    for part in parts[:-1]:
        if part not in current:
            current[part] = {}
        elif not isinstance(current[part], dict):
            raise TypeError(f"path component {part!r} is not a dictionary")
        current = current[part]

    current[parts[-1]] = deepcopy(value)

## Step 3 — Apply all updates to a deep copy

In [40]:
def update_dotted_paths(mapping, updates):
    result = deepcopy(mapping)
    for path, value in updates.items():
        set_dotted_path(result, path, value)
    return result

updated_configuration = update_dotted_paths(configuration, dotted_updates)
updated_configuration

{'database': {'host': 'db.internal',
  'port': 5432,
  'credentials': {'user': 'service', 'password': 'dev-password'}},
 'features': {'search': True, 'analytics': True}}

## Step 4 — Verify isolation and newly created paths

In [41]:
assert updated_configuration["database"]["host"] == "db.internal"
assert updated_configuration["database"]["credentials"]["user"] == "service"
assert updated_configuration["features"]["analytics"] is True
assert configuration["database"]["host"] == "localhost"
assert "analytics" not in configuration["features"]

updated_configuration["database"]["credentials"]["password"] = "changed"
assert configuration["database"]["credentials"]["password"] == "dev-password"

print(updated_configuration)

{'database': {'host': 'db.internal', 'port': 5432, 'credentials': {'user': 'service', 'password': 'changed'}}, 'features': {'search': True, 'analytics': True}}


## Step 5 — Handle a path collision

A path cannot descend through a scalar value. The function raises a `TypeError` instead of silently replacing the scalar with a dictionary.

In [42]:
try:
    update_dotted_paths(configuration, {"database.port.value": 123})
except TypeError as exc:
    print(exc)

path component 'port' is not a dictionary


## Solution discussion

This solution deep-copies the whole configuration for clarity and isolation. For extremely large structures, a path-copying technique may be more efficient, but it is also more complex.

# Problem 11 — Combine keyword-argument providers without silent collisions

Dictionary unpacking is convenient when calling functions:

```python
function(**options)
```

But ordinary dictionary merging silently lets the last provider win. For critical calls, duplicate keys from independent providers may indicate a configuration mistake.

Build a helper that combines providers and raises an error when the same key is supplied twice.

## Step 1 — A target function and independent providers

In [43]:
def connect(*, host, port, timeout=10, ssl=False):
    return {
        "host": host,
        "port": port,
        "timeout": timeout,
        "ssl": ssl,
    }

network_options = {"host": "db.example.com", "port": 5432}
security_options = {"ssl": True}
timeout_options = {"timeout": 20}

## Step 2 — Detect collisions while merging

We keep track of the provider that first supplied each key. This produces a useful error message.

In [44]:
def combine_unique_kwargs(*named_providers):
    combined = {}
    owners = {}

    for provider_name, provider in named_providers:
        for key, value in provider.items():
            if key in combined:
                raise ValueError(
                    f"duplicate keyword {key!r}: "
                    f"provided by {owners[key]!r} and {provider_name!r}"
                )
            combined[key] = value
            owners[key] = provider_name

    return combined

connection_kwargs = combine_unique_kwargs(
    ("network", network_options),
    ("security", security_options),
    ("timeout", timeout_options),
)

connection_kwargs

{'host': 'db.example.com', 'port': 5432, 'ssl': True, 'timeout': 20}

## Step 3 — Unpack the validated dictionary into the function

In [45]:
connection = connect(**connection_kwargs)

assert connection == {
    "host": "db.example.com",
    "port": 5432,
    "timeout": 20,
    "ssl": True,
}

connection

{'host': 'db.example.com', 'port': 5432, 'timeout': 20, 'ssl': True}

## Step 4 — Demonstrate collision detection

In [46]:
try:
    combine_unique_kwargs(
        ("network", network_options),
        ("override", {"port": 9999}),
    )
except ValueError as exc:
    print(exc)

duplicate keyword 'port': provided by 'network' and 'override'


## Solution discussion

Sometimes last-value-wins is exactly what we want. In other situations, duplicate keys should be treated as errors. The correct behavior depends on the meaning of the data.

# Problem 12 — Capstone: a versioned settings store

Build a small in-memory settings store that supports:

- reading an isolated snapshot;
- applying a patch;
- preserving version history;
- computing a diff for each commit;
- rolling back to an earlier version;
- ensuring callers cannot mutate stored history accidentally.

This problem combines updating, copying, patching, and difference computation.

## Step 1 — Define a commit record

A dataclass makes the stored information explicit.

In [47]:
@dataclass(frozen=True)
class Commit:
    version: int
    message: str
    state: dict
    difference: dict

## Step 2 — Design the store

The store keeps one internal current state and a list of commits. Every state saved into history is deeply copied.

The public `snapshot` method also returns a deep copy, so callers never receive direct access to internal mutable dictionaries.

In [48]:
class VersionedSettings:
    def __init__(self, initial):
        self._state = deepcopy(initial)
        self._commits = [
            Commit(
                version=0,
                message="initial state",
                state=deepcopy(initial),
                difference={
                    "added": deepcopy(initial),
                    "removed": {},
                    "changed": {},
                    "unchanged": {},
                },
            )
        ]

    @property
    def version(self):
        return self._commits[-1].version

    def snapshot(self):
        return deepcopy(self._state)

    def history(self):
        return deepcopy(self._commits)

    def commit_patch(self, patch, message):
        new_state, _ = apply_patch_with_inverse(self._state, patch)
        difference = dict_diff(self._state, new_state)
        new_version = self.version + 1

        self._state = new_state
        self._commits.append(
            Commit(
                version=new_version,
                message=message,
                state=deepcopy(new_state),
                difference=deepcopy(difference),
            )
        )
        return new_version

    def rollback(self, version, message=None):
        matches = [commit for commit in self._commits if commit.version == version]
        if not matches:
            raise ValueError(f"unknown version: {version}")

        target = matches[0].state
        difference = dict_diff(self._state, target)
        new_version = self.version + 1

        self._state = deepcopy(target)
        self._commits.append(
            Commit(
                version=new_version,
                message=message or f"rollback to version {version}",
                state=deepcopy(target),
                difference=deepcopy(difference),
            )
        )
        return new_version

## Step 3 — Create a store and make commits

In [49]:
store = VersionedSettings({
    "theme": "light",
    "font_size": 14,
    "notifications": True,
})

v1 = store.commit_patch(
    [
        ("set", "theme", "dark"),
        ("set", "font_size", 16),
    ],
    "enable dark theme",
)

v2 = store.commit_patch(
    [
        ("delete", "notifications"),
        ("set", "language", "en"),
    ],
    "simplify preferences",
)

print("Current version:", store.version)
print("Current state:", store.snapshot())

Current version: 2
Current state: {'theme': 'dark', 'font_size': 16, 'language': 'en'}


## Step 4 — Inspect history without exposing internal objects

In [50]:
history_copy = store.history()
for commit in history_copy:
    print(f"version={commit.version} message={commit.message!r}")
    print(" state:", commit.state)
    print(" diff:", commit.difference)

version=0 message='initial state'
 state: {'theme': 'light', 'font_size': 14, 'notifications': True}
 diff: {'added': {'theme': 'light', 'font_size': 14, 'notifications': True}, 'removed': {}, 'changed': {}, 'unchanged': {}}
version=1 message='enable dark theme'
 state: {'theme': 'dark', 'font_size': 16, 'notifications': True}
 diff: {'added': {}, 'removed': {}, 'changed': {'theme': {'old': 'light', 'new': 'dark'}, 'font_size': {'old': 14, 'new': 16}}, 'unchanged': {'notifications': True}}
version=2 message='simplify preferences'
 state: {'theme': 'dark', 'font_size': 16, 'language': 'en'}
 diff: {'added': {'language': 'en'}, 'removed': {'notifications': True}, 'changed': {}, 'unchanged': {'theme': 'dark', 'font_size': 16}}


Now mutate the returned history copy. The actual store should remain unchanged.

In [51]:
history_copy[-1].state["theme"] = "CORRUPTED"
assert store.snapshot()["theme"] == "dark"
print("Internal state remained safe:", store.snapshot())

Internal state remained safe: {'theme': 'dark', 'font_size': 16, 'language': 'en'}


## Step 5 — Roll back by creating a new version

Rollback should not erase history. Instead, it creates a new commit whose state matches an earlier version.

In [52]:
v3 = store.rollback(0, "restore original defaults")

assert v1 == 1
assert v2 == 2
assert v3 == 3
assert store.snapshot() == {
    "theme": "light",
    "font_size": 14,
    "notifications": True,
}

print("Rolled back state:", store.snapshot())
print("Current version:", store.version)

Rolled back state: {'theme': 'light', 'font_size': 14, 'notifications': True}
Current version: 3


## Step 6 — Confirm that old snapshots remain stable

In [53]:
all_history = store.history()
assert all_history[1].state == {
    "theme": "dark",
    "font_size": 16,
    "notifications": True,
}
assert all_history[2].state == {
    "theme": "dark",
    "font_size": 16,
    "language": "en",
}
assert all_history[3].state == {
    "theme": "light",
    "font_size": 14,
    "notifications": True,
}

print("All historical states are intact.")

All historical states are intact.


## Capstone discussion

This implementation demonstrates several best practices:

- internal mutable state is never returned directly;
- history stores deep copies rather than aliases;
- each commit includes a machine-readable difference;
- rollback is additive and auditable;
- patch operations are explicit rather than hidden inside arbitrary mutation code.

For a production system, additional concerns would include persistence, concurrency control, schema validation, access control, and storage limits.

# Additional guided challenges

The following exercises intentionally omit full implementations so that you can extend the notebook after studying the solved problems.

Each challenge includes a suggested decomposition.

## Challenge A — Nested three-way merge

Extend `three_way_merge` so nested dictionaries are merged recursively.

Suggested steps:

1. detect when the base, ours, and theirs values are all dictionaries;
2. recursively merge those dictionaries;
3. prefix nested conflict paths, such as `database.port`;
4. preserve additions and deletions;
5. write tests for conflicts at multiple depths.

## Challenge B — Patch preconditions

Add an optional expected old value to each patch operation:

```python
("set", "theme", "light", "dark")
```

The patch should fail when the current value is not the expected value.

Suggested steps:

1. define a clear operation format;
2. validate all preconditions before changing anything;
3. apply the operations only after validation succeeds;
4. return an inverse patch;
5. test that a failed patch leaves the input unchanged.

## Challenge C — Memory-conscious snapshots

The capstone deep-copies every complete state. Design an alternative that stores only differences.

Suggested steps:

1. store the initial full state once;
2. store one diff per commit;
3. rebuild a version by replaying diffs;
4. cache selected reconstructed versions;
5. compare time and memory trade-offs.

# Summary of best practices

1. Decide whether an operation should mutate its input or return a new dictionary.
2. Make overwrite precedence explicit and test it.
3. Remember that overwriting a key does not change its insertion position.
4. Use `dict(source)` to normalize mapping-like or pair-iterable update sources.
5. Do not assume replacement is the correct merge meaning for every field.
6. Use identity checks when investigating shared nested objects.
7. Use shallow copies for independent outer containers with intentionally shared values.
8. Use deep copies at isolation boundaries, snapshot boundaries, and history boundaries.
9. Represent deletions explicitly when building patches or merge algorithms.
10. Report conflicts instead of silently guessing a resolution.
11. Keep history immutable from the caller's point of view.
12. Verify solutions with assertions, invalid-input tests, and non-mutation checks.